In [ ]:
pip install transformers datasets evaluate -qq

# Question Answering Model Fine tuning

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import DefaultDataCollator
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer
from transformers import pipeline

In [ ]:
squad = load_dataset("rajpurkar/squad", split="train[:5000]")

In [ ]:
squad = squad.train_test_split(test_size=0.1)

In [ ]:
squad["train"][0]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [ ]:
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )
 
    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []
 
    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)
 
        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1
 
        # If the answer is not fully inside the context, label it (0, 0)
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)
 
            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)
 
    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [ ]:
tokenized_squad = squad.map(
    preprocess_function,
    batched=True,
    remove_columns=squad["train"].column_names
)

In [ ]:
data_collator = DefaultDataCollator()

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained("distilbert/distilbert-base-uncased")

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad["train"],
    eval_dataset=tokenized_squad["test"],
    processing_class=tokenizer,
    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("qa_model")
tokenizer.save_pretrained("qa_model")

In [ ]:
question = "How to make a jet engine"

context = """
Building a scale model of a jet engine requires a clear understanding of its core components and assembly order.Here is a structural breakdown of the major sections and assembly steps to help you build or describe your model.Core Sections of a Jet EngineA standard turbojet engine consists of five primary sections, arranged from front to back:Inlet (Intake): Funnels incoming air into the compressor.Compressor: Rows of spinning blades that squeeze and pressurize the air.Combustion Chamber (Combustor): Mixing zone where fuel injects into the pressurized air and ignites.Turbine: Rows of blades that catch the energy of escaping hot gases to spin the compressor shaft.Nozzle (Exhaust): Narrows the exiting hot air to create high-velocity thrust.Step-by-Step Model Assembly Order1. The Central Shaft (The Core)Function: Connects the turbine at the back to the compressor at the front.Model Step: Start with a central rod or axle. This acts as the structural spine for all rotating parts.
"""

In [ ]:
question_answerer = pipeline("question-answering", model="qa_model", tokenizer="qa_model")

In [ ]:
question_answerer(question=question, context=context)